In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb
import warnings
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')

# ==========================================
# 1. PATHS & DATA INGESTION
# ==========================================
base_path = '/workspaces/CECS-399-499/local_data/gold/'
silver_path = '/workspaces/CECS-399-499/local_data/silver/'

def get_master_data():
    print("Loading Gold Parquet files and Silver weather...")
    energy_feat = pd.read_parquet(f'{base_path}fact_energy_features_hourly.parquet')
    energy_load = pd.read_parquet(f'{base_path}fact_energy_load_hourly.parquet')
    time_dim = pd.read_parquet(f'{base_path}dim_time_hourly.parquet')
    
    df = pd.merge(energy_feat, energy_load, on=['time_key', 'source_id'])
    df = pd.merge(df, time_dim, left_on='time_key', right_on='time_id', how='left')
    
    weather_csv = pd.read_csv(f'{silver_path}tn_weighted_weather_21_25.csv')
    
    # Filter strictly to the weighted aggregate columns
    weighted_cols = [
        'timestamp', 'weighted_temperature_2m', 'weighted_relative_humidity_2m', 
        'weighted_precipitation', 'weighted_cloud_cover', 
        'weighted_wind_speed_10m', 'weighted_shortwave_radiation'
    ]
    weather_csv = weather_csv[weighted_cols]
    
    df['timestamp_join'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_localize(None)
    weather_csv['timestamp'] = pd.to_datetime(weather_csv['timestamp'], utc=True).dt.tz_localize(None)
    
    df = pd.merge(df, weather_csv, left_on='timestamp_join', right_on='timestamp', how='inner')
    return df.sort_values('timestamp_join').reset_index(drop=True)

df_master = get_master_data()

# ==========================================
# 2. FEATURE ENGINEERING & TARGET MAPPING
# ==========================================
print("Engineering raw demand and balance deltas...")
df_master['demand_delta_1h'] = df_master['actual_demand_mwh'].diff()
df_master['balance_delta_1h'] = df_master['balance_error'].diff()
df_master = df_master.dropna().reset_index(drop=True)

print("Mapping outage targets (Eagle1 + DOE 417)...")
eagle_dates = set(pd.to_datetime(pd.read_parquet(f'{base_path}fact_outage_daily.parquet')['date']).dt.date.unique())
doe_df = pd.read_csv(f'{silver_path}doe417_tennessee_cleaned_final.csv')
doe_date_col = [c for c in doe_df.columns if 'date' in c.lower()][0]
doe_dates = set(pd.to_datetime(doe_df[doe_date_col], format='mixed').dt.date.unique())

combined_outage_dates = eagle_dates.union(doe_dates)
df_master['event_date'] = pd.to_datetime(df_master['timestamp_join']).dt.date
df_master['target'] = df_master['event_date'].apply(lambda x: 1 if x in combined_outage_dates else 0)

# ==========================================
# 3. CHRONOLOGICAL SPLIT
# ==========================================
all_numeric = df_master.select_dtypes(include=[np.number]).columns.tolist()
exclude = ['time_id', 'time_key', 'source_id', 'year', 'quarter', 'month', 'hour_of_day', 'day_of_week', 'target']
FEATURES = [f for f in all_numeric if f not in exclude]

EXCLUDE_DRIVERS = ['month_sin', 'month_cos', 'hour_sin', 'hour_cos']

split_idx = int(len(df_master) * 0.6)
train_df = df_master.iloc[:split_idx].copy()
test_df = df_master.iloc[split_idx:].copy()

spw = (len(train_df) - train_df['target'].sum()) / train_df['target'].sum()

# ==========================================
# 4. TUNING & MODEL TRAINING
# ==========================================
param_grid = {
    'scale_pos_weight': [1.0, 1.5, 2.0, 2.5, spw], 
    'max_depth': [3, 4, 5, 6],                     
    'min_child_weight': [1, 3, 5, 7],              
    'subsample': [0.8, 0.9, 1.0],                  
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],      
    'gamma': [0, 0.1, 0.2, 0.3]                    
}

base_clf = xgb.XGBClassifier(
    n_estimators=200,      
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

print(f"\nStarting Randomized Search... (Baseline SPW: {spw:.2f})")
random_search = RandomizedSearchCV(
    estimator=base_clf,
    param_distributions=param_grid,
    n_iter=20,
    scoring='f1',          
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(train_df[FEATURES], train_df['target'])
best_clf = random_search.best_estimator_

print("\n--- OPTIMAL PARAMETERS FOUND ---")
for key, value in random_search.best_params_.items():
    print(f"{key}: {value}")

# Test Set Evaluation
test_probs = best_clf.predict_proba(test_df[FEATURES])[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

print("\n" + "="*45)
print(" VALIDATION: TEST SET PERFORMANCE ")
print("="*45)
cm = confusion_matrix(test_df['target'], test_preds)
print(f"True Negatives:  {cm[0][0]}")
print(f"False Positives: {cm[0][1]} (False Alarms)")
print(f"False Negatives: {cm[1][0]} (Missed Outages)")
print(f"True Positives:  {cm[1][1]} (Correct Hits)")
print("-" * 45)
print(classification_report(test_df['target'], test_preds))

# ==========================================
# 5. FULL INFERENCE & NATIVE SHAP EXTRACTION
# ==========================================
print("\nExtracting physical anomaly drivers via native XGBoost SHAP...")
df_master['anomaly_score'] = best_clf.predict_proba(df_master[FEATURES])[:, 1]
df_master['is_anomaly'] = (df_master['anomaly_score'] >= 0.5).astype(int)

# Use the tuned booster for SHAP extraction
booster = best_clf.get_booster()
dmatrix = xgb.DMatrix(df_master[FEATURES])
shap_contributions = booster.predict(dmatrix, pred_contribs=True)

# Drop the last column (bias)
shap_matrix = shap_contributions[:, :-1]

filtered_drivers = []
for i in range(len(df_master)):
    row_shap = pd.Series(shap_matrix[i], index=FEATURES)
    actionable_shap = row_shap.drop(labels=[f for f in EXCLUDE_DRIVERS if f in row_shap.index])
    top_3 = actionable_shap.abs().sort_values(ascending=False).index[:3].tolist()
    filtered_drivers.append(top_3)

drivers_df = pd.DataFrame(filtered_drivers, columns=['primary_driver', 'secondary_driver', 'tertiary_driver'])
df_master = pd.concat([df_master, drivers_df], axis=1)

# ==========================================
# 6. WAREHOUSE FINALIZATION
# ==========================================
warehouse_cols = [
    'time_key', 'source_id', 'timestamp_join', 'target', 
    'anomaly_score', 'is_anomaly', 'primary_driver', 
    'secondary_driver', 'tertiary_driver', 'demand_delta_1h', 'balance_delta_1h'
]
fact_anomaly_report = df_master[warehouse_cols].copy()

fact_anomaly_report.columns = [
    'time_key', 'source_id', 'timestamp', 'actual_outage_target',
    'anomaly_score', 'anomaly_flag', 'primary_driver', 'secondary_driver', 
    'tertiary_driver', 'raw_demand_delta', 'raw_balance_delta'
]

print("\n--- TOP DETECTED ANOMALIES (WAREHOUSE PREVIEW) ---")
print(fact_anomaly_report[fact_anomaly_report['anomaly_flag'] == 1].sort_values('anomaly_score', ascending=False).head(15).to_string(index=False))

Loading Gold Parquet files and Silver weather...
Engineering raw demand and balance deltas...
Mapping outage targets (Eagle1 + DOE 417)...

Starting Randomized Search... (Baseline SPW: 3.30)
Fitting 3 folds for each of 20 candidates, totalling 60 fits

--- OPTIMAL PARAMETERS FOUND ---
subsample: 0.8
scale_pos_weight: 3.3042979002624673
min_child_weight: 5
max_depth: 3
gamma: 0.2
colsample_bytree: 0.8

 VALIDATION: TEST SET PERFORMANCE 
True Negatives:  13138
False Positives: 2412 (False Alarms)
False Negatives: 24 (Missed Outages)
True Positives:  1920 (Correct Hits)
---------------------------------------------
              precision    recall  f1-score   support

           0       1.00      0.84      0.92     15550
           1       0.44      0.99      0.61      1944

    accuracy                           0.86     17494
   macro avg       0.72      0.92      0.76     17494
weighted avg       0.94      0.86      0.88     17494


Extracting physical anomaly drivers via native XGBoo

In [3]:
import pandas as pd
anomaly_detection = pd.read_parquet('/workspaces/CECS-399-499/local_data/gold/fact_anomaly_detection.parquet')
anomaly_detection.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43733 entries, 0 to 43732
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   time_key              43733 non-null  object        
 1   source_id             43733 non-null  object        
 2   timestamp             43733 non-null  datetime64[ns]
 3   actual_outage_target  43733 non-null  int64         
 4   anomaly_score         43733 non-null  float32       
 5   anomaly_flag          43733 non-null  int64         
 6   primary_driver        43733 non-null  object        
 7   secondary_driver      43733 non-null  object        
 8   tertiary_driver       43733 non-null  object        
 9   raw_demand_delta      43733 non-null  float64       
 10  raw_balance_delta     43733 non-null  float64       
dtypes: datetime64[ns](1), float32(1), float64(2), int64(2), object(5)
memory usage: 3.5+ MB
